# Player Matching Between FIFA and Transfermarkt

This notebook loads both player datasets, normalizes name formats, and matches records that likely refer to the same player.

In [ ]:
import pandas as pd
import re
import unicodedata

In [ ]:
fifa_path = "../Datasets/Football Players Data/fifa_players.csv"
tm_path = "../Datasets/Football Data from Transfermarkt/players.csv"

fifa = pd.read_csv(fifa_path, low_memory=False)
tm = pd.read_csv(tm_path, low_memory=False)

fifa = fifa.copy()
tm = tm.copy()

fifa["fifa_row_id"] = fifa.index
tm["tm_row_id"] = tm.index

print(f"FIFA rows: {len(fifa):,}")
print(f"Transfermarkt rows: {len(tm):,}")

In [ ]:
# Birth date distribution (day + month) before any merge.
fifa_birth_raw = fifa["birth_date"]
tm_birth_raw = tm["date_of_birth"]

# Parse dates with a fallback for mixed formats.
fifa_birth_dt = pd.to_datetime(fifa_birth_raw, errors="coerce")
tm_birth_dt = pd.to_datetime(tm_birth_raw, errors="coerce")

fifa_failed = fifa_birth_dt.isna()
tm_failed = tm_birth_dt.isna()
if fifa_failed.any():
    fifa_birth_dt.loc[fifa_failed] = pd.to_datetime(
        fifa_birth_raw.loc[fifa_failed], errors="coerce", dayfirst=True
    )
if tm_failed.any():
    tm_birth_dt.loc[tm_failed] = pd.to_datetime(
        tm_birth_raw.loc[tm_failed], errors="coerce", dayfirst=True
    )

fifa_day_month_dist = (
    pd.DataFrame({
        "month": fifa_birth_dt.dt.month,
        "day": fifa_birth_dt.dt.day
    })
    .dropna()
    .astype({"month": int, "day": int})
    .value_counts()
    .rename("fifa_count")
    .reset_index()
)

tm_day_month_dist = (
    pd.DataFrame({
        "month": tm_birth_dt.dt.month,
        "day": tm_birth_dt.dt.day
    })
    .dropna()
    .astype({"month": int, "day": int})
    .value_counts()
    .rename("tm_count")
    .reset_index()
)

day_month_distribution = (
    fifa_day_month_dist.merge(tm_day_month_dist, on=["month", "day"], how="outer")
    .fillna(0)
    .astype({"fifa_count": int, "tm_count": int})
)

# Order by FIFA frequency (highest first).
day_month_distribution_by_fifa = day_month_distribution.sort_values(
    ["fifa_count", "month", "day"], ascending=[False, True, True]
).reset_index(drop=True)

print(f"FIFA valid birth dates: {len(fifa_day_month_dist):,} unique day-month combinations")
print(f"Transfermarkt valid birth dates: {len(tm_day_month_dist):,} unique day-month combinations")

day_month_distribution_by_fifa.head(40)

In [ ]:
def normalize_name(value: str) -> str:
    if pd.isna(value):
        return ""
    s = str(value).strip().lower()

    # Remove accents/diacritics (e.g., Jose -> jose).
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))

    # Keep letters, numbers, and spaces only.
    s = re.sub(r"[^a-z0-9\s]", " ", s)

    # Remove common suffixes and collapse whitespace.
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# Use FIFA full_name when available, otherwise fallback to name.
fifa["name_source"] = fifa["full_name"].fillna(fifa["name"])
tm["name_source"] = tm["name"]

fifa["norm_name"] = fifa["name_source"].map(normalize_name)
tm["norm_name"] = tm["name_source"].map(normalize_name)

# Match only on normalized full name.
exact_matches = fifa.merge(
    tm,
    on="norm_name",
    how="inner",
    suffixes=("_fifa", "_tm")
)

exact_matches = exact_matches.drop_duplicates(subset=["fifa_row_id", "tm_row_id"])

result = exact_matches[[
    "fifa_row_id",
    "name_fifa",
    "full_name",
    "birth_date",
    "tm_row_id",
    "name_tm",
    "first_name",
    "last_name",
    "date_of_birth",
    "current_club_name",
    "market_value_in_eur"
 ]].sort_values(["name_fifa", "name_tm"]).reset_index(drop=True)

total_matches = len(result)
print(f"Total matched records (using normalized names): {total_matches:,}")
result.head(20)

In [ ]:
# Compare birth dates across matched records after standardizing date formats.
fifa_birth = pd.to_datetime(result["birth_date"], errors="coerce")
tm_birth = pd.to_datetime(result["date_of_birth"], errors="coerce")

# Retry failed parses with day-first interpretation for mixed formats.
fifa_failed = fifa_birth.isna()
tm_failed = tm_birth.isna()
if fifa_failed.any():
    fifa_birth.loc[fifa_failed] = pd.to_datetime(
        result.loc[fifa_failed, "birth_date"], errors="coerce", dayfirst=True
    )
if tm_failed.any():
    tm_birth.loc[tm_failed] = pd.to_datetime(
        result.loc[tm_failed, "date_of_birth"], errors="coerce", dayfirst=True
    )

birth_date_comparison = result[["name_fifa", "name_tm", "birth_date", "date_of_birth"]].copy()
birth_date_comparison["birth_date_fifa_std"] = fifa_birth.dt.date
birth_date_comparison["birth_date_tm_std"] = tm_birth.dt.date

valid_dates = (
    birth_date_comparison["birth_date_fifa_std"].notna()
    & birth_date_comparison["birth_date_tm_std"].notna()
)
same_birth_date_count = (
    birth_date_comparison.loc[valid_dates, "birth_date_fifa_std"]
    == birth_date_comparison.loc[valid_dates, "birth_date_tm_std"]
).sum()

print(f"Matched rows with valid dates in both datasets: {valid_dates.sum():,}")
print(f"Players with the same birth date in both datasets: {same_birth_date_count:,}")

birth_date_comparison.head(20)

In [ ]:
# Show matched players where standardized birth dates do not match.
date_mismatches = birth_date_comparison[
    valid_dates
    & (birth_date_comparison["birth_date_fifa_std"] != birth_date_comparison["birth_date_tm_std"])
][[
    "name_fifa",
    "name_tm",
    "birth_date",
    "date_of_birth",
    "birth_date_fifa_std",
    "birth_date_tm_std",
]].sort_values(["name_fifa", "name_tm"]).reset_index(drop=True)

print(f"Players with mismatched birth dates: {len(date_mismatches):,}")
date_mismatches.head(50)

In [ ]:
# Show non-matched birth dates where FIFA date is February 29.
fifa_229_mismatches = date_mismatches[
    (pd.to_datetime(date_mismatches["birth_date_fifa_std"], errors="coerce").dt.month == 2)
    & (pd.to_datetime(date_mismatches["birth_date_fifa_std"], errors="coerce").dt.day == 29)
][[
    "name_fifa",
    "name_tm",
    "birth_date",
    "date_of_birth",
    "birth_date_fifa_std",
    "birth_date_tm_std",
]].sort_values(["name_fifa", "name_tm"]).reset_index(drop=True)

print(f"Non-matched rows where FIFA date is 2-29: {len(fifa_229_mismatches):,}")
fifa_229_mismatches

In [ ]:
# Show matched records where date of birth is February 29 (in either dataset).
fifa_is_feb29 = (
    pd.to_datetime(birth_date_comparison["birth_date_fifa_std"], errors="coerce").dt.month.eq(2)
    & pd.to_datetime(birth_date_comparison["birth_date_fifa_std"], errors="coerce").dt.day.eq(29)
)
tm_is_feb29 = (
    pd.to_datetime(birth_date_comparison["birth_date_tm_std"], errors="coerce").dt.month.eq(2)
    & pd.to_datetime(birth_date_comparison["birth_date_tm_std"], errors="coerce").dt.day.eq(29)
)

feb29_matched_records = birth_date_comparison[
    fifa_is_feb29 | tm_is_feb29
][[
    "name_fifa",
    "name_tm",
    "birth_date",
    "date_of_birth",
    "birth_date_fifa_std",
    "birth_date_tm_std",
]].sort_values(["name_fifa", "name_tm"]).reset_index(drop=True)

print(f"Matched records with Feb 29 DOB (either dataset): {len(feb29_matched_records):,}")
print(f" - Feb 29 in FIFA date: {fifa_is_feb29.sum():,}")
print(f" - Feb 29 in Transfermarkt date: {tm_is_feb29.sum():,}")
print(f" - Feb 29 in both: {(fifa_is_feb29 & tm_is_feb29).sum():,}")

feb29_matched_records

In [ ]:
# Count FIFA non-matched records where birth date is February 29.
matched_fifa_ids = set(exact_matches["fifa_row_id"].unique())
fifa_not_matched = fifa[~fifa["fifa_row_id"].isin(matched_fifa_ids)].copy()

fifa_not_matched_birth = pd.to_datetime(fifa_not_matched["birth_date"], errors="coerce")
fifa_nm_failed = fifa_not_matched_birth.isna()
if fifa_nm_failed.any():
    fifa_not_matched_birth.loc[fifa_nm_failed] = pd.to_datetime(
        fifa_not_matched.loc[fifa_nm_failed, "birth_date"], errors="coerce", dayfirst=True
    )

fifa_not_matched_feb29 = fifa_not_matched[
    fifa_not_matched_birth.dt.month.eq(2) & fifa_not_matched_birth.dt.day.eq(29)
].copy()

print(f"FIFA not-matched records with birth date Feb 29: {len(fifa_not_matched_feb29):,}")
fifa_not_matched_feb29[["fifa_row_id", "name", "full_name", "birth_date"]].head(20)